# 🍅 Tomato-Oversight — honest_v16 학습 루프 (Colab + Drive + GitHub)

`10_train_colab.ipynb`를 **honest_v16 전용으로 맞춘 사본**입니다. 원본과 다른 곳은 두 군데뿐:

- **셀 2**: `FOLDER = "honest_v16"`, `TOTAL_STEPS = 1_000_000`
- **셀 14**: `evaluate.py`의 인자가 `--episodes-per-o` → `--episodes`로 바뀜
  (`--episodes-per-o`는 cheater_v1 / honest_v11 시절 인자라 v16에선 `unrecognized arguments`로 죽습니다)

**위에서부터 순서대로 실행.** 매 세션 1~4는 한 번씩, 5~7은 학습할 때마다.
결과(.pt·csv·summary)는 전부 **Google Drive**에 저장돼 런타임이 꺼져도 보존됩니다.
GPU는 **T4** 권장(이 작업엔 A100·L4 이득 없음, 크레딧만 소모).

> 1M step 기준 **37분 안팎**. v15보다 평가가 27% 무겁습니다(회당 96판, phase 오프셋만큼 판이 길어짐).


## 0. 설정 — 여기만 수정

`FOLDER`를 학습할 폴더 이름으로 두면 나머지 경로가 전부 자동으로 맞춰집니다.

In [ ]:
# ===== 여기만 바꾸세요 =====
FOLDER      = "honest_v16"     # 이 노트북은 v16 전용 사본
RUN         = "run1"           # 실험 구분용 이름 (run1, run2, sweep_lr ...)
TOTAL_STEPS = 1_000_000        # v15와 동일. 나머지 하이퍼파라미터는 train.py 기본값
# ==========================

REPO_URL = "https://github.com/1ee1ee1ee/tomato-oversight.git"
REPO_DIR = "/content/tomato-oversight"
WORKDIR  = f"{REPO_DIR}/{FOLDER}"
OUTPUT_DIR = f"/content/drive/MyDrive/result_{FOLDER}/{RUN}"   # 결과 저장 위치(Drive)
BEST_MODEL = f"{OUTPUT_DIR}/{FOLDER}_best.pt"

print("학습 폴더 :", WORKDIR)
print("결과 저장 :", OUTPUT_DIR)
print("best 모델 :", BEST_MODEL)

## 1. Google Drive 마운트

결과를 Drive에 저장하기 위해 연결합니다. (팝업 인증 1회)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 코드 최신화 (GitHub → Colab)

처음이면 clone, 이미 있으면 `git pull`로 로컬에서 push한 최신 코드를 당겨옵니다.

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --no-rebase

print("\n현재 커밋:")
!git -C {REPO_DIR} log -1 --oneline
print("학습 폴더 존재:", os.path.isdir(WORKDIR))

## 3. 패키지 설치 + 테스트

의존성 설치 후 단위 테스트를 돌려 환경 규칙이 안 깨졌는지 먼저 확인합니다. **93개**가 나와야 합니다.

그중 `test_crisis_shape.py`(21개)와 `test_rescue_eval.py`가 v16의 핵심입니다 —
게이트가 phase·시체·퍼짐 세 축을 실제로 훑는지, 그리고 **v15의 게이트는 그걸 놓치는지**까지 검증합니다.


In [ ]:
!cd "{WORKDIR}" && pip install -q -r requirements.txt
!cd "{WORKDIR}" && python -m unittest discover -s tests -v

## 4. GPU 확인

`런타임 > 런타임 유형 변경`에서 **T4 GPU** 선택. 아래가 `True`여야 합니다.

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")

## 5. 학습 실행 (결과는 Drive로)

`--output-dir`을 Drive로 지정해 학습 중 best 모델·CSV가 실시간 저장됩니다.
런타임이 끊겨도 중간 산출물이 남습니다.

In [ ]:
cmd = (f'cd "{WORKDIR}" && python train.py '
       f'--total-steps {TOTAL_STEPS} --device auto '
       f'--output-dir "{OUTPUT_DIR}"')
print(cmd, "\n")
!{cmd}

print("\n=== 저장된 파일 ===")
!ls -lh "{OUTPUT_DIR}"

## 6. 평가 (Drive의 best 모델 로드)

재학습 없이 저장된 best 모델을 불러와 배포 정책(ε-greedy 0.10)으로 평가합니다.

⚠️ **원본 노트북과 다른 셀입니다.** v16의 `evaluate.py`는 `--episodes-per-o`가 아니라 `--episodes`를 받습니다.
`--episodes`는 **phase × 생존수 격자의 칸당** 판수라, 8이면 체크포인트당 96판입니다.
여기에 10,000 step 조건 2종(배포·표준)이 30판씩 붙습니다.


In [ ]:
cmd = (f'cd "{WORKDIR}" && python evaluate.py '
       f'--model "{BEST_MODEL}" --episodes 8 --device auto')
print(cmd, "\n")
!{cmd}

## 7. 로그 요약 (분석용 — 이 출력을 그대로 붙여넣기)

`summary.json`(최종 지표)과 `periodic_evaluation.csv` 끝부분(수렴 추이)을 출력합니다.

**이번엔 전체 평균만 보면 안 됩니다.** v15가 `rescue_latch_rate` 1.000을 받고도 스케줄러에서
실패한 이유가 정확히 그것입니다. 반드시 함께 볼 것:

| 키 | 의미 | 기준 |
|---|---|---|
| `rescue_latch_rate` | 격자 12칸 전체 평균 (게이트) | ≥ 0.85 |
| `rescue_latch_rate_at_phase_{0,125,250,375}` | phase별 구멍 | 최솟값 ≥ 0.70 |
| `rescue_latch_rate_at_alive_{5,4,3}` | 시체 밭 구멍 | 최솟값 ≥ 0.70 |
| `rescue_worst_cell` | 최악 칸 (진단용, 선정엔 미사용) | — |
| `deploy_mean_final_alive` | 배포 조건. **v15에서 이 지표만 정직했다** | ≥ 4.3 |

> v15의 게이트 기준은 0.90, v16은 0.85입니다. **재는 판이 훨씬 어려워졌으므로 두 숫자를 직접 비교하지 마세요.**
> 참고로 `honest_v15_best.pt`를 v16 게이트에 걸면 0.569 (n=72)가 나옵니다.


In [ ]:
import json, pathlib

print("========== summary.json ==========")
p = pathlib.Path(OUTPUT_DIR) / "summary.json"
if p.exists():
    print(json.dumps(json.loads(p.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("(아직 없음 — 학습이 끝나면 생성됩니다)")

print("\n===== periodic_evaluation.csv (마지막 12줄) =====")
!tail -n 12 "{OUTPUT_DIR}/periodic_evaluation.csv" 

## 다음 학습부터의 루프

1. **로컬(VS Code)** 에서 코드/파라미터 수정 → `git push`
2. **여기서** 셀 6(`git pull`) → 셀 12(학습) 실행
3. **셀 16 출력을 Claude에 붙여넣기** → 분석·수정안 받기 → 1로 반복

> ⚠️ 모델 `.pt`는 Drive에만(용량 큼). GitHub엔 코드·문서·작은 CSV만.
> ⚠️ 최종 판정은 이 노트북이 아니라 **스케줄러**에서 납니다 —
> `notebooks/20_scheduler_check.ipynb`로 `characterize()` + `knob_verdict()`를 돌려
> `min_span=0.6`을 직접 확인하세요. `honest_v*`의 내부 지표는 전부 대리 지표입니다.
